Load in and resample the impulse responses using Librosa
Create h5 file to contain all

In [1]:
import librosa
import h5py
import numpy as np
from pathlib import Path

In [2]:
brir_path = Path('/om/user/francl/Room_Simulator_20181115_Rebuild/')

In [3]:
room_paths = [x for x in brir_path.glob('Expanded*') if '6x4y' not in x.as_posix()]

In [35]:
room_paths[0].stem

'Expanded_HRIRdist140-5deg_elev_az_room8x5y5z_materials8wall14floor16ciel'

In [8]:
irs = list(eg_room.glob('*r.wav'))

NameError: name 'eg_room' is not defined

In [29]:
locations = np.unique([ir.stem.split('_')[2:][0] for ir in irs], return_counts=False) 

In [31]:
loc_dict = {loc: ['_'.join(ir.stem.split('_')[:-1]) for ir in irs if loc in ir.stem] for loc in locations}

In [10]:
## Room irs eg
room = '1.40x1.40y2.00z'

room_irs = loc_dict[room]

n_sources = len(room_irs)

ir_paths = [(path + '_l.wav', path + '_r.wav') for path in room_irs]
# print(ir_paths)

# Get tensor of irs as "kernel"

goal_r = 50_000
ir_dur = 0.5 # dur in seconds 
### USE NUMPY, LOCATIONS INSTEAD OF SOURCES
ir_kernel = np.zeros((n_sources, 2, int(goal_r * ir_dur)))
azimuth_kernel = np.zeros((n_sources, 1))
elevation_kernel = np.zeros((n_sources, 1))

for ix,  (left_ir, right_ir) in enumerate(ir_paths):
    ### USE LIBROSA COMMAND HERE
    left_ir_up, sr = librosa.load(eg_room/left_ir, sr = goal_r,                    
                                res_type='soxr_vhq',
                                dtype='float32')
    right_ir_up, sr = librosa.load(eg_room/right_ir, sr = goal_r,                    
                                res_type='soxr_vhq',
                                dtype='float32')

    ### ADD THESE TO N x 2 x T array
    ir_kernel[ix][0] = left_ir_up
    ir_kernel[ix][1] = right_ir_up
    
    ### ADD TO two N x 1 Azimuth and Elevation arrays as int
    elev, az = left_ir.split('_')[:2]
    azimuth_kernel[ix] = int(az[:-2])
    elevation_kernel[ix] = int(elev[:-4])


[[ 75.]
 [160.]
 [ 65.]
 ...
 [130.]
 [255.]
 [300.]]


In [9]:
with h5py.File('test_dummmy.h5', 'w') as h5:
    for room in room_paths:
        room_name = room.stem
        irs = list(room.glob('*r.wav'))
        locations = np.unique([ir.stem.split('_')[2:][0] for ir in irs], return_counts=False)
        loc_dict = {loc: ['_'.join(ir.stem.split('_')[:-1]) for ir in irs if loc in ir.stem] for loc in locations}
        roomgroup = h5.create_group(room_name)
        for listener_loc in locations:
            loc_irs = loc_dict[listener_loc]
            n_sources = len(loc_irs)
            ir_paths = [(path + '_l.wav', path + '_r.wav') for path in loc_irs]
            goal_r = 50_000
            ir_dur = 0.5
            ir_kernel = np.zeros((n_sources, 2, int(goal_r * ir_dur)))
            azimuth_kernel = np.zeros((n_sources, 1))
            elevation_kernel = np.zeros((n_sources, 1))
            for ix,  (left_ir, right_ir) in enumerate(ir_paths):
                left_ir_up, sr = librosa.load(room/left_ir, sr = goal_r,                    
                                            res_type='soxr_vhq',
                                            dtype='float32')
                right_ir_up, sr = librosa.load(room/right_ir, sr = goal_r,                    
                                            res_type='soxr_vhq',
                                            dtype='float32')

                ### ADD THESE TO N x 2 x T array
                ir_kernel[ix][0] = left_ir_up
                ir_kernel[ix][1] = right_ir_up
                
                ### ADD TO two N x 1 Azimuth and Elevation arrays as int
                elev, az = left_ir.split('_')[:2]
                azimuth_kernel[ix] = int(az[:-2])
                elevation_kernel[ix] = int(elev[:-4])
            roomgroup.create_dataset(f"{listener_loc}/irs", data=ir_kernel.astype('float32'))
            roomgroup.create_dataset(f"{listener_loc}/azimuth", data=azimuth_kernel.astype('float32'))
            roomgroup.create_dataset(f"{listener_loc}/elevation", data=elevation_kernel.astype('float32'))

In [10]:
h5 = h5py.File('test_dummmy.h5', 'r')

In [15]:
h5['Expanded_HRIRdist140-5deg_elev_az_room10x10y4z_materials11wall22floor20ciel/4.47x8.60y2.00z'].keys()

<KeysViewHDF5 ['azimuth', 'elevation', 'irs']>